# baseline v3

이 베이스라인 코드는 `사전학습 모델 로드`, `배치 학습`, `파인튜닝`, `양자화`, `PEFT` 등이 적용된 버전입니다.

Colab의 GPU 환경에서 개발되었습니다.
- 런타임 - 런타임 유형 변경 - GPU로 변경(T4 GPU 등)



# 환경 준비

개발 환경에 필요한 라이브러리 버전을 고정하고 최신 버전으로 라이브러리를 업데이트합니다.

- 아래 셀 실행
- 실행 완료 후 런타임 - 세션 다시 시작

In [ ]:
!pip -q install git+https://github.com/huggingface/transformers accelerate
!pip -q install --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio
!pip -q install "peft>=0.13.2" "bitsandbytes==0.46.1" datasets pillow pandas --upgrade

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [1]:
import torch
print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())

Torch version: 2.10.0+cu128
CUDA version: 12.8
cuDNN version: 91002


# 데이터 준비

개발에 필요한 데이터를 준비합니다.

- train.csv, train 폴더
- test.csv, test 폴더
- sample_submission.csv

본 베이스라인은 colab에서 구글 드라이브를 마운트하여 사용합니다.

데이터를 압축 해제하는데 몇 분 정도의 시간이 소요됩니다.

#### 실습 참고 내용

    챕터 2-2 합성 데이터 실습
    - 구글 드라이브 마운트 : drive()

In [2]:
# 구글드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# 압축 해제
!unzip "/content/drive/My Drive/A131_dataset/2026-ssafy-ai-15-2.zip" -d "/content/"

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
  inflating: /content/train/train_0074.jpg  
  inflating: /content/train/train_0075.jpg  
  inflating: /content/train/train_0076.jpg  
  inflating: /content/train/train_0077.jpg  
  inflating: /content/train/train_0078.jpg  
  inflating: /content/train/train_0079.jpg  
  inflating: /content/train/train_0080.jpg  
  inflating: /content/train/train_0081.jpg  
  inflating: /content/train/train_0082.jpg  
  inflating: /content/train/train_0083.jpg  
  inflating: /content/train/train_0084.jpg  
  inflating: /content/train/train_0085.jpg  
  inflating: /content/train/train_0086.jpg  
  inflating: /content/train/train_0087.jpg  
  inflating: /content/train/train_0088.jpg  
  inflating: /content/train/train_0089.jpg  
  inflating: /content/train/train_0090.jpg  
  inflating: /content/train/train_0091.jpg  
  inflating: /content/train/train_0092.jpg  
  inflating: /content/train/train_0093.jpg  
  inflating: /content/train/train_0094.jpg  
  inflating: /conte

In [54]:
!ls -lh /content/2026-ssafy-ai-15-2.zip

ls: cannot access '/content/2026-ssafy-ai-15-2.zip': No such file or directory


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 라이브러리, 데이터, 설정

In [3]:
import os, re, math, random
import pandas as pd
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
from typing import Dict, List, Any
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup,
    get_cosine_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm import tqdm

# 디바이스 GPU 우선 사용 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# 이미지 로드 시 픽셀 제한 해제
Image.MAX_IMAGE_PIXELS = None

# 사전 학습 모델 정의
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
IMAGE_SIZE = 384
MAX_NEW_TOKENS = 8
SEED = 42
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# 데이터셋 로드
train_df = pd.read_csv("/content/train.csv")
test_df  = pd.read_csv("/content/test.csv")
dev_df = pd.read_csv("/content/dev.csv")

# # 학습데이터 200개만 추출
# train_df = train_df.sample(n=200, random_state=SEED).reset_index(drop=True)

resp_cols = ['answer1', 'answer2', 'answer3', 'answer4', 'answer5']

# 빈 응답(NaN)에 대응하는 안전한 필터링 함수
def get_safe_majority(row):
    counts = row[resp_cols].value_counts()
    if len(counts) > 0 and counts.iloc[0] >= 4:
        return counts.index[0]
    return np.nan

# 함수 적용
dev_df['answer'] = dev_df.apply(get_safe_majority, axis=1)
dev_clean = dev_df.dropna(subset=['answer']).copy()

# 데이터 통합
common_cols = ['id', 'path', 'question', 'a', 'b', 'c', 'd', 'answer']
final_train_df = pd.concat([train_df[common_cols], dev_clean[common_cols]], axis=0).reset_index(drop=True)

# 검증셋 분리 (9:1)
split = int(len(final_train_df) * 0.9)
train_subset = final_train_df.iloc[:split]
valid_subset = final_train_df.iloc[split:]
print(f"✅ 학습 데이터: {len(train_subset)}개 / 자체 검증 데이터: {len(valid_subset)}개")


Device: cuda
✅ 학습 데이터: 5377개 / 자체 검증 데이터: 598개


# 모델, Processor

7.5GB 정도의 모델 다운로드가 진행됩니다. 10~20분 정도가 소요됩니다.

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - LoRA 구현 : LoraConfig()

In [6]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
!pip install flash-attn --no-build-isolation

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 153.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
ERROR: Operation cancelled by user

In [4]:
# LLaVA 모델 로드 (Flash Attention 2 적용)
MODEL_ID ="Qwen/Qwen2.5-VL-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16 # A100이므로 bfloat16 사용

)

# 프로세서
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=256 * 28 * 28,
    max_pixels=1024 * 28 * 28,
    # min_pixels=IMAGE_SIZE*IMAGE_SIZE,
    # max_pixels=IMAGE_SIZE*IMAGE_SIZE,
    trust_remote_code=True,
)

# 사전학습 모델
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    #attn_implementation="flash_attention_2"
)

# 양자화 모델로 로드
base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable()

# LoRA 세팅
lora_config = LoraConfig(
    #r=8,
    r=32,
    #lora_alpha=16,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
)

# PEFT 모델 생성
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

trainable params: 95,178,752 || all params: 8,387,345,408 || trainable%: 1.1348


# 프롬프트 템플릿

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - 프롬프트 템플릿 : convert_to_chatml(), formatting_prompts_func()

In [5]:
# 개선된 SYSTEM_INSTRUCT
SYSTEM_INSTRUCT = (
    "You are a professional image analysis expert specializing in waste sorting and recycling. "
    "Your goal is to analyze the provided image and select the most accurate answer from the given options. "
    "Respond ONLY with the lowercase letter (a, b, c, or d). Do not include any other text, reasoning, or punctuation."
)

# 개선된 프롬프트 함수
def build_mc_prompt_v2(question, a, b, c, d):
    return (
        "### Image Analysis Task\n"
        f"Question: {question}\n\n"
        "### Options\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "### Instruction\n"
        "1. Observe the material, texture, and labels of the object in the image carefully.\n"
        "2. Choose the correct option from above.\n"
        "3. Output only the single letter (a, b, c, or d).\n\n"
        "Answer: "
    )

# Custom Dataset, Collator

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - TensorDataset()

    챕터 5-2 데이터 생성 및 파인튜닝 (향후 학습 분량)
    - IntentDataset()

In [6]:
from torchvision import transforms

# Qwen2.5-VL 맞춤형 증강 전략
vqa_transform = transforms.Compose([
    # 1. 무작위 회전 (-15도 ~ 15도) : 삐뚤게 찍힌 사진에 대한 적응력 향상
    transforms.RandomRotation(degrees=15, fill=255), # 배경은 흰색으로 채움

    # 2. 색상 왜곡 (밝기, 대비, 채도) : 실내/실외 다양한 조명 환경 적응
    # Qwen은 질감을 잘 보므로 대비(Contrast) 증강이 특히 효과적입니다.
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),

    # 3. 선명도 조절 : 초점이 살짝 나간 사진 보정
    transforms.RandomAdjustSharpness(sharpness_factor=2, p=0.5),

    # 4. 가우시안 블러 (10% 확률) : 흔들린 사진에 대한 강인함(Robustness) 확보
    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=(3, 5), sigma=(0.1, 2.0))
    ], p=0.1),
])

# ==========================================
# 1. 커스텀 데이터셋
# ==========================================
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")

        if self.train:
            img = vqa_transform(img)

            # 5. 좌우 반전 (글자가 중요한 질문이 없다면 50% 확률로 적용)
            # 만약 재활용 마크 안의 '글자'를 읽어야 하는 데이터라면 이 줄은 빼는게 좋습니다.
            if random.random() > 0.5:
                img = img.transpose(Image.FLIP_LEFT_RIGHT)

        q = str(row["question"])
        a, b, c, d = str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])

        # 💡 수정 1: 개선된 _v2 프롬프트 함수 호출
        user_text = build_mc_prompt_v2(q, a, b, c, d)

        ans = str(row["answer"]).strip().lower()

        messages = [
            {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
            {"role":"user","content":[
                {"type":"image","image":img},
                {"type":"text","text":user_text}
            ]}
        ]
        if self.train:
            gold = str(row["answer"]).strip().lower()
            messages.append({"role":"assistant","content":[{"type":"text","text":gold}]})

        return {"messages": messages, "image": img, "train": self.train, "answer": ans}

# ==========================================
# 2. 데이터 콜레이터 (Loss 마스킹 적용)
# ==========================================
@dataclass
class DataCollator:
    processor: Any

    def __call__(self, batch):
        texts, prompt_texts, images, answers = [], [], [], [] # answers 추가
        is_train = batch[0]["train"]

        for sample in batch:
            messages = sample["messages"]
            images.append(sample["image"])
            answers.append(sample["answer"]) # 정답 문자열 보관

            if is_train:
                text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
                texts.append(text)
                prompt_msg = [m for m in messages if m['role'] != 'assistant']
                prompt_text = self.processor.apply_chat_template(prompt_msg, tokenize=False, add_generation_prompt=True)
                prompt_texts.append(prompt_text)
            else:
                text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
                texts.append(text)

        enc = self.processor(text=texts, images=images, padding=True, return_tensors="pt")

        if is_train:
            prompt_enc = self.processor(text=prompt_texts, images=images, padding=True, return_tensors="pt")
            labels = enc["input_ids"].clone()
            for i in range(len(batch)):
                p_len = prompt_enc["attention_mask"][i].sum().item()
                labels[i, :p_len] = -100
            labels[enc["attention_mask"] == 0] = -100
            enc["labels"] = labels

        # 💡 리스트 형태의 answers를 포함해서 리턴 (나중에 pop으로 뺄 것임)
        enc["answers"] = answers
        return enc

# DataLoader

#### 실습 참고 내용

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 데이터로더 정의 : DataLoader()

In [7]:
# 학습은 train=True, 검증(Loss 계산용)도 정답이 필요하므로 train=True 유지
train_ds = VQAMCDataset(train_subset, processor, train=True)
valid_ds = VQAMCDataset(valid_subset, processor, train=True)

# A100에 맞게 배치 사이즈 2로 상향 (메모리 부족 시 1로 변경)
BATCH_SIZE = 4
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=DataCollator(processor), num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=DataCollator(processor), num_workers=0)

In [59]:
# ==========================================
# 1. 실험용 미니 데이터셋 생성 (딱 100장!)
# ==========================================
print("🧪 실험 모드: 딱 100장만 뽑아서 테스트를 시작합니다.")
test_subset_size = 100
valid_subset_size = 20

# 무작위로 섞어서 추출
mini_train_df = final_train_df.sample(n=test_subset_size, random_state=42).reset_index(drop=True)
mini_valid_df = final_train_df.sample(n=valid_subset_size, random_state=7).reset_index(drop=True)

# 데이터로더 재생성 (Batch Size 2 정도로 가볍게)
mini_train_ds = VQAMCDataset(mini_train_df, processor, train=True)
mini_valid_ds = VQAMCDataset(mini_valid_df, processor, train=False) # 추론 테스트용이므로 False

mini_train_loader = DataLoader(mini_train_ds, batch_size=2, shuffle=True, collate_fn=DataCollator(processor))
mini_valid_loader = DataLoader(mini_valid_ds, batch_size=2, shuffle=False, collate_fn=DataCollator(processor))

# ==========================================
# 2. 초간단 1 에포크 학습 테스트
# ==========================================
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
scaler = torch.amp.GradScaler('cuda', enabled=True)

print("🚀 1단계: 학습 루프가 에러 없이 도는지 확인 중...")
for batch in tqdm(mini_train_loader, desc="Mini Train"):
    batch = {k:v.to(device) for k,v in batch.items() if k != 'train'}

    with torch.amp.autocast('cuda', dtype=torch.bfloat16):
        outputs = model(**batch)
        loss = outputs.loss

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad()

print("✅ 학습 루프 통과!")

# ==========================================
# 3. 실제 정답(a, b, c, d) 추출 테스트
# ==========================================
print("\n🚀 2단계: 실제 모델이 글자를 뱉어내는지 확인 중...")
model.eval()
processor.tokenizer.padding_side = 'left' # 추론 시 필수!

correct = 0
total = 0

with torch.no_grad():
    # 5개 샘플만 직접 확인
    for i in range(5):
        sample = mini_valid_ds[i]
        img = sample["image"]
        messages = sample["messages"]
        true_ans = sample["answer"]

        # 추론용 텍스트 준비
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[img], return_tensors="pt").to(device)

        # 모델 답변 생성
        out_ids = model.generate(**inputs, max_new_tokens=5, do_sample=False)

        # 입력 프롬프트 제외하고 정답만 디코딩
        input_len = inputs["input_ids"].shape[1]
        output_text = processor.decode(out_ids[0][input_len:], skip_special_tokens=True).strip().lower()

        # 선지 추출 함수 적용
        pred = output_text[0] if output_text else "n/a"

        print(f"[{i+1}] 실제정답: {true_ans} | 모델예측: {pred} (원본: {output_text})")
        if pred == true_ans:
            correct += 1
        total += 1

print(f"\n🎯 미니 테스트 결과: {correct}/{total} 적중!")
print("✅ 모든 시스템 정상 작동 확인! 이제 본 학습으로 넘어가셔도 좋습니다.")

🧪 실험 모드: 딱 100장만 뽑아서 테스트를 시작합니다.
🚀 1단계: 학습 루프가 에러 없이 도는지 확인 중...


Mini Train: 100%|██████████| 50/50 [03:32<00:00,  4.24s/it]


✅ 학습 루프 통과!

🚀 2단계: 실제 모델이 글자를 뱉어내는지 확인 중...
[1] 실제정답: b | 모델예측: d (원본: d)
[2] 실제정답: a | 모델예측: a (원본: a)
[3] 실제정답: d | 모델예측: d (원본: d)
[4] 실제정답: d | 모델예측: d (원본: d)
[5] 실제정답: b | 모델예측: b (원본: b)

🎯 미니 테스트 결과: 4/5 적중!
✅ 모든 시스템 정상 작동 확인! 이제 본 학습으로 넘어가셔도 좋습니다.


In [ ]:
import torch
from tqdm import tqdm
from torch.utils.data import DataLoader

# ==========================================
# 1. 실험용 미니 데이터셋 생성 (딱 100장!)
# ==========================================
print("🧪 실험 모드: 딱 100장만 뽑아서 테스트를 시작합니다.")
test_subset_size = 100
valid_subset_size = 20

# 무작위로 섞어서 추출 (위 셀에서 선언한 final_train_df 사용)
mini_train_df = final_train_df.sample(n=test_subset_size, random_state=42).reset_index(drop=True)
mini_valid_df = final_train_df.sample(n=valid_subset_size, random_state=7).reset_index(drop=True)

# 데이터로더 재생성 (V100 메모리를 고려해 Batch Size 2로 가볍게)
mini_train_ds = VQAMCDataset(mini_train_df, processor, train=True)
mini_valid_ds = VQAMCDataset(mini_valid_df, processor, train=False)

mini_train_loader = DataLoader(mini_train_ds, batch_size=2, shuffle=True, collate_fn=DataCollator(processor))
mini_valid_loader = DataLoader(mini_valid_ds, batch_size=2, shuffle=False, collate_fn=DataCollator(processor))


# ==========================================
# 2. 초간단 1 에포크 학습 테스트
# ==========================================
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
scaler = torch.amp.GradScaler('cuda', enabled=True)

print("\n🚀 1단계: 학습 루프가 에러 없이 도는지 확인 중...")
for batch in tqdm(mini_train_loader, desc="Mini Train"):

    # 💡 핵심 해결책 적용: Collator에서 넘어온 문자열 리스트(answers) 및 train 플래그 제거
    batch.pop("answers", None)
    batch = {k: v.to(device) for k, v in batch.items() if k != 'train'}

    # 💡 V100 환경이므로 bfloat16이 아닌 float16 사용
    with torch.amp.autocast('cuda', dtype=torch.float16):
        outputs = model(**batch)
        loss = outputs.loss

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)

print("✅ 학습 루프 통과!")


# ==========================================
# 3. 실제 정답(a, b, c, d) 추출 테스트
# ==========================================
print("\n🚀 2단계: 실제 모델이 글자를 뱉어내는지 확인 중...")
model.eval()
processor.tokenizer.padding_side = 'left' # 추론 시에는 좌측 패딩 필수!

correct = 0
total = 0

with torch.no_grad():
    # 5개 샘플만 직접 확인 (Dataset에서 직접 1개씩 꺼내오므로 Collator 에러 무관)
    for i in range(5):
        sample = mini_valid_ds[i]
        img = sample["image"]
        messages = sample["messages"]
        true_ans = sample["answer"]

        # 추론용 텍스트 및 이미지 준비
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[img], return_tensors="pt").to(device)

        # 💡 추론 시에도 float16 캐스팅 적용 (메모리 절약 및 속도 향상)
        with torch.amp.autocast('cuda', dtype=torch.float16):
            out_ids = model.generate(**inputs, max_new_tokens=5, do_sample=False)

        # 입력 프롬프트 부분을 제외하고 모델이 생성한 정답만 디코딩
        input_len = inputs["input_ids"].shape[1]
        output_text = processor.decode(out_ids[0][input_len:], skip_special_tokens=True).strip().lower()

        # 선지 추출 함수 적용 (첫 글자만 가져오도록 안전망 설정)
        pred = output_text[0] if output_text else "n/a"

        print(f"[{i+1}] 실제정답: {true_ans} | 모델예측: {pred} (원본: {output_text})")
        if pred == true_ans:
            correct += 1
        total += 1

print(f"\n🎯 미니 테스트 결과: {correct}/{total} 적중!")
print("✅ 모든 시스템 정상 작동 확인! 이제 본 학습으로 넘어가셔도 좋습니다.")

🧪 실험 모드: 딱 100장만 뽑아서 테스트를 시작합니다.

🚀 1단계: 학습 루프가 에러 없이 도는지 확인 중...


Mini Train: 100%|██████████| 50/50 [03:33<00:00,  4.27s/it]


✅ 학습 루프 통과!

🚀 2단계: 실제 모델이 글자를 뱉어내는지 확인 중...
[1] 실제정답: b | 모델예측: b (원본: b)
[2] 실제정답: a | 모델예측: a (원본: a)
[3] 실제정답: d | 모델예측: d (원본: d)
[4] 실제정답: d | 모델예측: d (원본: d)
[5] 실제정답: b | 모델예측: b (원본: b)

🎯 미니 테스트 결과: 5/5 적중!
✅ 모든 시스템 정상 작동 확인! 이제 본 학습으로 넘어가셔도 좋습니다.


# fine-tuning

- 200개만 학습 : 10~20분 소요

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - 모델 정의 : SimpleMLP(), SequentialMLP()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [9]:
model = model.to(device)
GRAD_ACCUM = 16
EPOCHS = 2

# 옵티마이저, 학습 스케줄러
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
num_training_steps = EPOCHS * math.ceil(len(train_loader)/GRAD_ACCUM) # 에포크 3으로 가정
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(num_training_steps*0.1), num_training_steps=num_training_steps)

# 스케일러 (PyTorch 최신 문법으로 업데이트)
scaler = torch.amp.GradScaler('cuda', enabled=True)

# 학습 시작 (3 Epoch)
for epoch in range(EPOCHS):
    model.train()
    processor.tokenizer.padding_side = 'right' # 학습 시 우측 패딩
    running = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} [train]", unit="batch")

    for step, batch in enumerate(progress_bar, start=1):
        batch.pop("answers", None)
        batch = {k:v.to(device) for k,v in batch.items()}

        with torch.amp.autocast('cuda', dtype=torch.float16):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        running += loss.item()

        if step % GRAD_ACCUM == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

            avg_loss = running / GRAD_ACCUM
            progress_bar.set_postfix({"loss": f"{avg_loss:.4f}"})
            running = 0.0

    # 검증 루프 (Validation Loss 확인)
    model.eval()
    processor.tokenizer.padding_side = 'left'
    val_loss = 0.0
    val_steps = 0

    with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.float16):
        for vb in tqdm(valid_loader, desc=f"Epoch {epoch+1} [valid]", unit="batch"):
            vb.pop("answers", None)
            vb = {k:v.to(device) for k,v in vb.items()}
            val_loss += model(**vb).loss.item()
            val_steps += 1
    print(f"🎯 [Epoch {epoch+1}] Valid Loss: {val_loss/val_steps:.4f}")

    # 💡 수정 4: 에포크별로 저장 폴더 이름 명확히 설정 (3b -> 7b)
    CHECKPOINT_DIR = f"/content/qwen2_5_vl_7b_lora_epoch_{epoch+1}"
    model.save_pretrained(CHECKPOINT_DIR)
    processor.save_pretrained(CHECKPOINT_DIR)
    print(f"💾 Saved Checkpoint: {CHECKPOINT_DIR}")

# 최종 모델 저장
SAVE_DIR = "/content/qwen2_5_vl_7b_lora_best1"
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print(f"🎉 Final Model Saved: {SAVE_DIR}")

Epoch 1 [train]:   1%|          | 15/1345 [01:56<2:51:14,  7.73s/batch]/tmp/ipykernel_4417/903450376.py:35: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()
Epoch 1 [valid]: 100%|██████████| 150/150 [05:30<00:00,  2.20s/batch]


🎯 [Epoch 1] Valid Loss: 8.0221
💾 Saved Checkpoint: /content/qwen2_5_vl_7b_lora_epoch_1


Epoch 2 [train]:   2%|▏         | 26/1345 [03:28<2:55:56,  8.00s/batch, loss=0.0065]


KeyboardInterrupt: 

In [10]:
import torch
from peft import PeftModel
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from tqdm.auto import tqdm
import pandas as pd
from PIL import Image

# 1. 경로 설정 (어제 저장된 Epoch 1 폴더)
checkpoint_path = "/content/qwen2_5_vl_7b_lora_epoch_1"
base_model_id = "Qwen/Qwen2.5-VL-7B-Instruct"

print("🚀 모델 로딩 중... (시간이 조금 걸릴 수 있습니다)")

# 2. 베이스 모델 및 어댑터(LoRA) 불러오기
# A100 환경에 맞춰 16비트로 로드합니다.
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
model = PeftModel.from_pretrained(base_model, checkpoint_path)
processor = AutoProcessor.from_pretrained(checkpoint_path)

# 3. 추론을 위한 핵심 설정
model.eval()
processor.tokenizer.padding_side = 'left' # 🚨 생성(Generate) 시 필수!
device = "cuda" if torch.cuda.is_available() else "cpu"

# 4. 정답 추출 함수 (기존 함수 재사용)
def extract_choice(text: str) -> str:
    text = text.strip().lower()
    if not text: return "a"
    # 첫 글자가 a, b, c, d 중 하나라면 바로 반환
    if text[0] in ["a", "b", "c", "d"]:
        return text[0]
    # 문장 안에서 찾는 로직
    tokens = text.split()
    for tok in tokens:
        if tok in ["a", "b", "c", "d"]:
            return tok
    return "a"

# 5. 추론 루프
preds = []
print("🔍 테스트 데이터 추론 시작...")

with torch.no_grad():
    for i in tqdm(range(len(test_df)), desc="Inference"):
        row = test_df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")

        # 학습 때와 동일한 프롬프트 구성 (build_mc_prompt_v2가 선언되어 있어야 함)
        user_text = build_mc_prompt_v2(row["question"], row["a"], row["b"], row["c"], row["d"])

        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
            {"role": "user", "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": user_text}
            ]}
        ]

        # 템플릿 적용 및 토큰화
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[img], return_tensors="pt").to(device)

        # 답변 생성
        out_ids = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            eos_token_id=processor.tokenizer.eos_token_id
        )

        # 모델이 새로 생성한 토큰만 추출
        input_len = inputs["input_ids"].shape[1]
        output_text = processor.decode(out_ids[0][input_len:], skip_special_tokens=True).strip()

        # 결과 저장
        preds.append(extract_choice(output_text))

# 6. 결과 저장
submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv("/content/submission_epoch1.csv", index=False)

print("\n✅ 추론 완료! /content/submission_epoch1.csv 파일을 확인하세요.")

🚀 모델 로딩 중... (시간이 조금 걸릴 수 있습니다)


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

🔍 테스트 데이터 추론 시작...


Inference:   0%|          | 0/5074 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



✅ 추론 완료! /content/submission_epoch1.csv 파일을 확인하세요.


# inference

30분~1시간 소요

#### 실습 참고 내용

    챕터4-1 RAG 기반 Customer Service AI 에이전트 개발
    - 데이터 파서 : langchain_core.output_parsers(), StrOutputParser()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [61]:
from peft import PeftModel

# 1. 모델 및 프로세서 경로 설정 (학습 시 저장했던 경로)
# 가장 성적이 좋았던 폴더나 마지막 에포크 폴더를 지정하세요.
model_path = "/content/qwen2_5_vl_7b_lora_best1"

# 2. 베이스 모델과 어댑터(LoRA) 로드
print("모델 로드 중...")
# base_model_id는 처음에 설정했던 "Qwen/Qwen2.5-VL-7B-Instruct"입니다.
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)
model = PeftModel.from_pretrained(model, model_path)
processor = AutoProcessor.from_pretrained(model_path)
device = "cuda" if torch.cuda.is_available() else "cpu"

print("✅ 모델 로드 완료! 이제 추론을 시작합니다.")

# 데이터 파서 : 모델의 응답에서 선지를 추출
def extract_choice(text: str) -> str:
    text = text.strip().lower()

    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines:
        return "a"
    last = lines[-1]
    if last in ["a", "b", "c", "d"]:
        return last

    tokens = last.split()
    for tok in tokens:
        if tok in ["a", "b", "c", "d"]:
            return tok
    return "a"

# 추론을 위해 모든 레이어 활성화
model.eval()
preds = []

for i in tqdm(range(len(test_df)), desc="Inference", unit="sample"):
    row = test_df.iloc[i]
    img = Image.open(row["path"]).convert("RGB")

    # 💡 수정 3: build_mc_prompt_v2 로 함수명 수정
    user_text = build_mc_prompt_v2(row["question"], row["a"], row["b"], row["c"], row["d"])

    messages = [
        {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
        {"role":"user","content":[
            {"type":"image","image":img},
            {"type":"text","text":user_text}
        ]}
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[img], return_tensors="pt").to(device)

    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=2, do_sample=False,
                                 eos_token_id=processor.tokenizer.eos_token_id)
    output_text = processor.batch_decode(out_ids, skip_special_tokens=True)[0]
    preds.append(extract_choice(output_text))

# 제출 파일 생성
submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv("/content/submission.csv", index=False)
print("Saved /content/submission.csv")

모델 로드 중...


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# 모델 응답 예시
print(output_text)

system
You are a helpful visual question answering assistant. Answer using exactly one letter among a, b, c, or d. No explanation.
user
사진에 보이는 재활용 가능한 아이템은 무엇인가요?
(a) 플라스틱 숟가락
(b) 유리병
(c) 종이 포장지
(d) 금속 캔

정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요.
assistant
b
